# Gemma 2 2B OCR Error Corrector — Fine-tuning

Fine-tune `google/gemma-2-2b-it` on synthetic + real OCR error pairs for handwriting OCR post-processing.

1. Runtime → Change runtime type → **T4 GPU**
2. Run all cells
3. Checkpoint saved to Google Drive

Training approach:
- Freeze all layers except last 2 transformer blocks + lm_head (~10% trainable)
- Synthetic data: 60+ sentence corpus with OCR-like corruptions (m/rn, d/cl, etc.)
- Real data: IAM handwriting dataset run through Surya OCR
- 3 epochs, lr=2e-4, batch_size=4

In [ ]:
# Install dependencies
!pip install -q torch transformers accelerate datasets surya-ocr

# Login to HuggingFace (needed for gated Gemma model)
from huggingface_hub import login
from google.colab import userdata
login(token=userdata.get('HF_TOKEN'))

In [ ]:
# Check GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')

## Step 1: Generate Synthetic Training Data

In [ ]:
import random

# OCR-like character substitutions
CHAR_SUBSTITUTIONS = {
    "O": "0", "0": "O", "I": "1", "1": "I", "l": "1",
    "S": "5", "5": "S", "B": "8", "8": "B", "Z": "2", "2": "Z",
    "G": "6", "6": "G", "A": "4", "4": "A", "T": "7", "7": "T",
    "g": "9", "9": "g", "E": "3", "3": "E",
    "m": "rn", "rn": "m", "n": "ri", "u": "v", "v": "u",
    "c": "e", "e": "c", "a": "o", "o": "a",
    "d": "cl", "cl": "d", "h": "b", "b": "h",
    "w": "vv", "f": "t", "t": "f",
    "i": "j", "j": "i",
}

def corrupt_text(text, error_rate=0.15):
    chars = list(text)
    for i in range(len(chars)):
        if random.random() < error_rate:
            error_type = random.choice(["substitute", "delete", "duplicate", "swap", "space"])
            if error_type == "substitute" and chars[i] in CHAR_SUBSTITUTIONS:
                chars[i] = CHAR_SUBSTITUTIONS[chars[i]]
            elif error_type == "delete" and len(chars) > 1:
                chars[i] = ""
            elif error_type == "duplicate":
                chars[i] = chars[i] * 2
            elif error_type == "swap" and i < len(chars) - 1:
                chars[i], chars[i + 1] = chars[i + 1], chars[i]
            elif error_type == "space":
                if chars[i] == " ":
                    chars[i] = ""
                else:
                    chars[i] = chars[i] + " "
    return "".join(chars)

SENTENCE_CORPUS = [
    "Hello my name is", "How are you doing today",
    "The quick brown fox jumps over the lazy dog",
    "Please sign your name here", "Thank you very much",
    "Dear Sir or Madam", "Yours sincerely",
    "Best regards", "To whom it may concern",
    "123 Main Street", "456 Oak Avenue", "789 Elm Boulevard",
    "New York NY 10001", "Los Angeles CA 90001",
    "San Francisco California", "Chicago Illinois",
    "January 15 2024", "March 3 1995", "December 25 2000",
    "Date of birth", "Social Security Number",
    "Phone 555 123 4567", "Account number 987654",
    "Patient name", "Date of service", "Diagnosis code",
    "Signature of authorized representative",
    "I hereby certify that the above information is correct",
    "Policy number", "Claim reference",
    "The meeting is scheduled for next Tuesday",
    "Please review the attached document",
    "I would like to request a refund",
    "The total amount due is", "Payment received with thanks",
    "Notes from the interview", "Action items from today",
    "Reminder to call back tomorrow morning",
    "Pick up groceries on the way home",
    "Happy birthday to you", "Congratulations on your achievement",
    "Weather forecast calls for rain", "Temperature is 72 degrees",
    "John Smith", "Mary Johnson", "Robert Williams",
    "Jennifer Davis", "Michael Brown", "Sarah Wilson",
    "Alexander Hamilton", "Benjamin Franklin",
    "Elizabeth Warren", "Theodore Roosevelt",
    "Sid", "Hi this is Sid", "My name is Sid",
]

# Generate synthetic pairs
synthetic_pairs = []
for text in SENTENCE_CORPUS:
    for _ in range(10):
        corrupted = corrupt_text(text, error_rate=0.15)
        if corrupted != text:
            synthetic_pairs.append((corrupted, text))

print(f"Synthetic pairs: {len(synthetic_pairs)}")
print(f"Examples:")
for i in range(5):
    c, t = synthetic_pairs[i]
    print(f"  '{c}' -> '{t}'")

## Step 2: Generate Real OCR Pairs from IAM Dataset

In [ ]:
# Load IAM handwriting dataset and run Surya OCR to get real error pairs
from datasets import load_dataset
from PIL import Image
from surya.foundation import FoundationPredictor
from surya.recognition import RecognitionPredictor
from surya.detection import DetectionPredictor

print("Loading IAM dataset...")
iam = load_dataset("Teklia/IAM-line", split="train")
print(f"IAM samples: {len(iam)}")

print("Loading Surya models...")
foundation = FoundationPredictor(device="cuda")
det_predictor = DetectionPredictor(device="cuda")
rec_predictor = RecognitionPredictor(foundation)

# Run Surya on IAM images to collect real OCR errors
real_pairs = []
max_samples = 500  # limit for speed on free Colab

for i in range(min(max_samples, len(iam))):
    img = iam[i]["image"]
    ground_truth = iam[i]["text"].strip()

    if not ground_truth or len(ground_truth) < 3:
        continue

    if img.mode != "RGB":
        img = img.convert("RGB")

    try:
        results = rec_predictor([img], det_predictor=det_predictor)
        ocr_text = " ".join(
            line.text.strip() for line in results[0].text_lines
            if line.text.strip() and line.confidence < 0.99
        )

        if ocr_text and ocr_text != ground_truth:
            real_pairs.append((ocr_text, ground_truth))
    except Exception as e:
        continue

    if (i + 1) % 100 == 0:
        print(f"  Processed {i+1}/{max_samples}, collected {len(real_pairs)} error pairs")

print(f"\nReal OCR pairs: {len(real_pairs)}")
if real_pairs:
    print("Examples:")
    for i in range(min(5, len(real_pairs))):
        c, t = real_pairs[i]
        print(f"  '{c[:60]}' -> '{t[:60]}'")

## Step 3: Combine Data and Create Dataset

In [ ]:
from torch.utils.data import Dataset, DataLoader

PROMPT_TEMPLATE = "Correct the OCR errors in the following text:\n{input}\n\nCorrected:\n{output}"

class OCRCorrectionDataset(Dataset):
    def __init__(self, pairs, tokenizer, max_length=128):
        self.pairs = pairs
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        corrupted, clean = self.pairs[idx]
        prompt = PROMPT_TEMPLATE.format(input=corrupted, output=clean)
        encoding = self.tokenizer(
            prompt, truncation=True, max_length=self.max_length,
            padding="max_length", return_tensors="pt",
        )
        input_ids = encoding["input_ids"].squeeze(0)
        attention_mask = encoding["attention_mask"].squeeze(0)
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": input_ids.clone(),
        }

# Combine synthetic + real pairs (oversample real pairs)
all_pairs = list(synthetic_pairs)
if real_pairs:
    oversample = max(1, len(synthetic_pairs) // len(real_pairs))
    all_pairs.extend(real_pairs * oversample)
    print(f"Real pairs oversampled {oversample}x: {len(real_pairs)} -> {len(real_pairs) * oversample}")

random.shuffle(all_pairs)
print(f"Total training pairs: {len(all_pairs)}")

## Step 4: Load Gemma 2 2B and Fine-tune

In [ ]:
# Free Surya models from GPU memory before loading Gemma
import gc
del foundation, det_predictor, rec_predictor
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory free: {torch.cuda.mem_get_info()[0] / 1024**3:.1f} GB")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, get_linear_schedule_with_warmup
import os

model_name = "google/gemma-2-2b-it"
device = "cuda"

print(f"Loading {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load in float16 to fit on T4 (16GB VRAM)
model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.float16
).to(device)

# Freeze all except last 2 layers + lm_head
for param in model.parameters():
    param.requires_grad = False

layers = model.model.layers
for layer in layers[-2:]:
    for param in layer.parameters():
        param.requires_grad = True

for param in model.lm_head.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

In [ ]:
# Create dataset and dataloader
dataset = OCRCorrectionDataset(all_pairs, tokenizer, max_length=128)
loader = DataLoader(dataset, batch_size=4, shuffle=True)

# Optimizer and scheduler
num_epochs = 3
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=2e-4,
)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=10,
    num_training_steps=num_epochs * len(loader),
)

print(f"Dataset size: {len(dataset)}")
print(f"Batches per epoch: {len(loader)}")
print(f"Total training steps: {num_epochs * len(loader)}")

In [ ]:
# Training loop
model.train()
for epoch in range(num_epochs):
    total_loss = 0
    for batch_idx, batch in enumerate(loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

        if (batch_idx + 1) % 50 == 0:
            print(f"  Epoch {epoch+1}, batch {batch_idx+1}/{len(loader)}, loss={loss.item():.4f}")

    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch+1}/{num_epochs}, Avg Loss: {avg_loss:.4f}")

print("Training complete!")

## Step 5: Test the Corrector

In [ ]:
# Test the fine-tuned corrector
model.eval()

test_cases = [
    "to this is sed",          # "Hi this is Sid" (from TrOCR)
    "My named 35.00 Sign",     # "My name is Sid" (from Surya)
    "my name",                 # "my name" (already correct)
    "Helo Wrld",               # "Hello World"
    "7he qvick hrovvn tox",    # "The quick brown fox"
    "P1ease si9n yovr narne",  # "Please sign your name"
    "established belgished by the last of",  # real Surya output
]

for ocr_text in test_cases:
    prompt = f"Correct the OCR errors in the following text:\n{ocr_text}\n\nCorrected:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=64, do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

    full = tokenizer.decode(outputs[0], skip_special_tokens=True)
    marker = "Corrected:\n"
    if marker in full:
        corrected = full.split(marker)[-1].strip().split("\n")[0].strip()
    else:
        corrected = full

    print(f"  '{ocr_text}' -> '{corrected}'")

## Step 6: Save Checkpoint

In [ ]:
# Save to local
save_path = "ocr-corrector"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model saved to {save_path}")

# Save to Google Drive
from google.colab import drive
drive.mount('/content/drive')

import shutil
drive_path = "/content/drive/MyDrive/ocr-corrector"
shutil.copytree(save_path, drive_path, dirs_exist_ok=True)
print(f"Copied to Google Drive: {drive_path}")

In [ ]:
# Optional: Push to HuggingFace Hub
# model.push_to_hub("your-username/ocr-corrector-gemma-2-2b")
# tokenizer.push_to_hub("your-username/ocr-corrector-gemma-2-2b")